In [1]:

import os
import json
import requests

from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

LENS_TOKEN = os.getenv("LENS_API_TOKEN")
LENS_BASE_URL = "https://api.lens.org"

if not LENS_TOKEN:
    raise RuntimeError("LENS_API_TOKEN is missing")

mcp = FastMCP("Lens-Intelligence-Server")


def lens_post(endpoint, payload):
    """Common authenticated Lens API request."""

    response = requests.post(
        f"{LENS_BASE_URL}/{endpoint}",
        headers={
            "Authorization": f"Bearer {LENS_TOKEN}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=45
    )

    if response.status_code == 429:
        raise RuntimeError(
            "Lens API rate limit reached. Retry later."
        )

    response.raise_for_status()

    if response.status_code == 204:
        return {"message": "No results found"}

    return response.json()


@mcp.tool()
def search_patents(
    query_json: dict,
    size: int = 20,
    offset: int = 0,
    group_by_family: bool = False
) -> dict:
    """
    Search Lens patent records.

    query_json must contain a valid Lens patent query DSL object.
    Use fields such as title, abstract, claim,
    applicant.name and application_reference.date.

    Returns patent records and available metadata.
    """

    if not 1 <= size <= 100:
        raise ValueError("Size must be between 1 and 100")

    if offset < 0:
        raise ValueError("Offset cannot be negative")

    payload = {
        "query": query_json,
        "size": size,
        "from": offset,
        "include": [
            "lens_id",
            "biblio",
            "doc_key"
        ]
    }

    if group_by_family:
        payload["group_by"] = "SIMPLE_FAMILY"

    return lens_post("patent/search", payload)


@mcp.tool()
def get_patent_details(lens_id: str) -> dict:
    """
    Retrieve an individual patent using its Lens ID.
    """

    if not lens_id.strip():
        raise ValueError("Lens ID is required")

    response = requests.get(
        f"{LENS_BASE_URL}/patent/{lens_id}",
        headers={
            "Authorization": f"Bearer {LENS_TOKEN}"
        },
        timeout=45
    )

    response.raise_for_status()
    return response.json()


@mcp.tool()
def search_scholarly_works(
    query_json: dict,
    size: int = 20,
    offset: int = 0
) -> dict:
    """
    Search Lens scholarly publications.
    query_json must follow the Lens scholarly query schema.
    """

    if not 1 <= size <= 100:
        raise ValueError("Size must be between 1 and 100")

    if offset < 0:
        raise ValueError("Offset cannot be negative")

    payload = {
        "query": query_json,
        "size": size,
        "from": offset
    }

    return lens_post("scholarly/search", payload)


@mcp.tool()
def get_competitor_patents(
    company_name: str,
    technology_query: str,
    start_date: str,
    end_date: str,
    size: int = 50
) -> dict:
    """
    Retrieve competitor patents within a technology
    and application-date range.

    Applicant name matching uses the Lens applicant.name field.
    """

    if not company_name.strip():
        raise ValueError("Company name is required")

    query_json = {
        "bool": {
            "must": [
                {
                    "match": {
                        "applicant.name": company_name
                    }
                },
                {
                    "match": {
                        "abstract": technology_query
                    }
                },
                {
                    "range": {
                        "application_reference.date": {
                            "gte": start_date,
                            "lte": end_date
                        }
                    }
                }
            ]
        }
    }

    return search_patents(
        query_json=query_json,
        size=size
    )


if __name__ == "__main__":
    mcp.run(transport="streamable-http")


ModuleNotFoundError: No module named 'mcp'